# HR Attrition — Production Preprocessing for XGBoost
**Pipeline:** Protected Attribute Isolation → Ordinal/OHE Encoding → HuggingFace Sentiment → Stratified Split → Class Weight Calculation

Target: `Attrition_Target` (1 = left, 0 = stayed). Highly imbalanced; handled via `scale_pos_weight`.

## 0. Imports & Config

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from transformers import pipeline
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

# ── Global Config ─────────────────────────────────────────────────────────────
RANDOM_STATE    = 42
TEST_SIZE       = 0.20
TARGET_COL      = 'Attrition_Target'
PROTECTED_COLS  = ['Age', 'Gender', 'Marital_Status']

# ── Data Paths ────────────────────────────────────────────────────────────────
PATH_EMPLOYEE     = 'employee_data.csv'
PATH_COMPENSATION = 'compensation_data.csv'
PATH_SURVEY       = 'survey_results.csv'
PATH_ATTRITION    = 'attrition_data.csv'
PATH_MARKET       = 'market_benchmarks.csv'

# ── Encoding Config ───────────────────────────────────────────────────────────
ORDINAL_MAPS = {
    'Education_Level': {
        'High School': 1,
        'Associate':   2,
        'Bachelor':    3,
        'Master':      4,
        'PhD':         5
    }
}
NOMINAL_COLS     = ['Department', 'Employment_Type', 'Work_Location']
TEXT_COL         = 'Feedback_Comments'
SENTIMENT_COL    = 'Feedback_Sentiment'
SENTIMENT_MODEL  = 'distilbert-base-uncased-finetuned-sst-2-english'
SENTIMENT_BATCH  = 32

# ── Logging helper ────────────────────────────────────────────────────────────
def log_df(df, label, show_dtypes=False, show_sample=False):
    """Print a consistent snapshot of a dataframe at any pipeline stage."""
    sep = "─" * 60
    print(f"\n{sep}")
    print(f"  📋 {label}")
    print(sep)
    print(f"  Shape   : {df.shape[0]:,} rows × {df.shape[1]} columns")
    print(f"  Columns ({df.shape[1]}):")
    for col in df.columns:
        dtype  = str(df[col].dtype)
        nulls  = df[col].isnull().sum()
        null_s = f"  ⚠ {nulls:,} nulls" if nulls > 0 else ""
        print(f"    • {col:<35} [{dtype}]{null_s}")
    if show_dtypes:
        print(f"\n  Dtype summary:")
        for dtype, count in df.dtypes.value_counts().items():
            print(f"    {dtype}: {count} column(s)")
    if show_sample:
        print(f"\n  First 2 rows:")
        print(df.head(2).to_string(index=False))
    print(sep)

print("✅ Config loaded.")
print(f"   Random state    : {RANDOM_STATE}")
print(f"   Test size       : {TEST_SIZE:.0%}")
print(f"   Target column   : {TARGET_COL}")
print(f"   Protected cols  : {PROTECTED_COLS}")
print(f"   Ordinal cols    : {list(ORDINAL_MAPS.keys())}")
print(f"   Nominal cols    : {NOMINAL_COLS}")
print(f"   Text column     : {TEXT_COL}  →  will become  {SENTIMENT_COL}")
print(f"   Sentiment model : {SENTIMENT_MODEL}")

/home/kav/anaconda3/envs/athena/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


✅ Config loaded.
   Random state    : 42
   Test size       : 20%
   Target column   : Attrition_Target
   Protected cols  : ['Age', 'Gender', 'Marital_Status']
   Ordinal cols    : ['Education_Level']
   Nominal cols    : ['Department', 'Employment_Type', 'Work_Location']
   Text column     : Feedback_Comments  →  will become  Feedback_Sentiment
   Sentiment model : distilbert-base-uncased-finetuned-sst-2-english


## 1. Load Data

In [2]:
# ── Step 1: Load raw files ────────────────────────────────────────────────────
employee     = pd.read_csv(PATH_EMPLOYEE)
compensation = pd.read_csv(PATH_COMPENSATION)
survey       = pd.read_csv(PATH_SURVEY)
attrition    = pd.read_csv(PATH_ATTRITION)
market       = pd.read_csv(PATH_MARKET)

print("📂 Raw files loaded:")
print(f"   employee_data      : {employee.shape[0]:,} rows  | cols: {list(employee.columns)}")
print(f"   compensation_data  : {compensation.shape[0]:,} rows  | cols: {list(compensation.columns)}")
print(f"   survey_results     : {survey.shape[0]:,} rows  | cols: {list(survey.columns)}")
print(f"   attrition_data     : {attrition.shape[0]:,} rows  | cols: {list(attrition.columns)}")
print(f"   market_benchmarks  : {market.shape[0]:,} rows  | cols: {list(market.columns)}")

# ── Step 2: Encode target ─────────────────────────────────────────────────────
# Attrition_Status is 'Yes'/'No'. We map → 1/0.
# Attrition_Date is dropped: it has NaNs for all stayers (8,192 rows) by design
# and would leak information (you can't know exit date at prediction time).
print("\n🔢 Encoding target variable...")
print(f"   Attrition_Status raw values : {attrition['Attrition_Status'].value_counts().to_dict()}")
print(f"   Attrition_Date nulls        : {attrition['Attrition_Date'].isnull().sum():,}  "
      f"(nulls = employees who stayed — column dropped to avoid leakage)")
attrition[TARGET_COL] = (attrition['Attrition_Status'] == 'Yes').astype(int)
attrition = attrition[['Employee_ID', TARGET_COL]]
print(f"   After encoding → kept cols  : {list(attrition.columns)}")
print(f"   Target distribution         : {attrition[TARGET_COL].value_counts().to_dict()}")

# ── Step 3: Derive Compa_Ratio ────────────────────────────────────────────────
# Compa_Ratio does NOT exist as a raw column anywhere.
# It is derived as: employee Base_Salary / external market Benchmark_Salary
# for the same Role + Work_Location combination.
# Role and Work_Location are used as join keys here and will be DROPPED afterwards —
# Role has 33 unique values (too high cardinality for OHE, no ordinal rank),
# and its salary signal is fully captured by Compa_Ratio once derived.
print("\n📊 Deriving Compa_Ratio via market benchmark join...")
comp_with_benchmark = compensation.merge(
    employee[['Employee_ID', 'Role', 'Work_Location']], on='Employee_ID'
).merge(
    market.rename(columns={'Location': 'Work_Location'}),
    on=['Role', 'Work_Location'], how='left'
)
comp_with_benchmark['Compa_Ratio'] = (
    comp_with_benchmark['Base_Salary'] / comp_with_benchmark['Benchmark_Salary']
)
nulls_cr = comp_with_benchmark['Compa_Ratio'].isnull().sum()
print(f"   Rows after join             : {comp_with_benchmark.shape[0]:,}")
print(f"   Compa_Ratio nulls           : {nulls_cr}  {'✅ clean' if nulls_cr == 0 else '⚠ check unmatched roles/locations'}")
print(f"   Compa_Ratio range           : [{comp_with_benchmark['Compa_Ratio'].min():.3f}, "
      f"{comp_with_benchmark['Compa_Ratio'].max():.3f}]  "
      f"(1.0 = exactly at market rate)")

# ── Step 4: Master merge ──────────────────────────────────────────────────────
print("\n🔗 Merging all sources on Employee_ID...")
before_cols = list(employee.columns)
df = (
    employee
    .merge(survey[['Employee_ID', 'Job_Satisfaction', 'Work_Life_Balance',
                   'Management_Support', 'Career_Development',
                   'Engagement_Level', 'Feedback_Comments']], on='Employee_ID')
    .merge(comp_with_benchmark[['Employee_ID', 'Base_Salary', 'Bonus',
                                 'Stock_Options', 'Total_Compensation',
                                 'Compa_Ratio']], on='Employee_ID')
    .merge(attrition, on='Employee_ID')
)
print(f"   employee cols               : {before_cols}")
print(f"   + survey added              : ['Job_Satisfaction', 'Work_Life_Balance', 'Management_Support', "
      f"'Career_Development', 'Engagement_Level', 'Feedback_Comments']")
print(f"   + compensation added        : ['Base_Salary', 'Bonus', 'Stock_Options', 'Total_Compensation', 'Compa_Ratio']")
print(f"   + attrition added           : ['{TARGET_COL}']")
print(f"   Shape after full merge      : {df.shape}")
print(f"   Row count preserved         : {'✅ yes' if len(df) == len(employee) else '⚠ mismatch — check for duplicate keys'}")

# ── Step 5: Drop non-feature columns ─────────────────────────────────────────
drop_cols = ['Employee_ID', 'Role']
print(f"\n🗑  Dropping non-feature columns: {drop_cols}")
print(f"   Employee_ID : unique string identifier, zero predictive value, causes object dtype error")
print(f"   Role        : 33 unique values — too high for OHE, no ordinal rank.")
print(f"                 Its salary signal is already captured by Compa_Ratio.")
print(f"                 Department already represents organisational grouping.")
df = df.drop(columns=drop_cols)
print(f"   Shape after drop            : {df.shape}")

log_df(df, "MASTER DATAFRAME — post-merge, pre-protected-split", show_sample=True)

📂 Raw files loaded:
   employee_data      : 10,000 rows  | cols: ['Employee_ID', 'Age', 'Gender', 'Department', 'Role', 'Tenure_Years', 'Employment_Type', 'Education_Level', 'Marital_Status', 'Work_Location']
   compensation_data  : 10,000 rows  | cols: ['Employee_ID', 'Base_Salary', 'Bonus', 'Stock_Options', 'Total_Compensation']
   survey_results     : 10,000 rows  | cols: ['Employee_ID', 'Job_Satisfaction', 'Work_Life_Balance', 'Management_Support', 'Career_Development', 'Engagement_Level', 'Feedback_Comments']
   attrition_data     : 10,000 rows  | cols: ['Employee_ID', 'Attrition_Status', 'Attrition_Date']
   market_benchmarks  : 297 rows  | cols: ['Role', 'Location', 'Benchmark_Salary', 'Data_Source']

🔢 Encoding target variable...
   Attrition_Status raw values : {'No': 8192, 'Yes': 1808}
   Attrition_Date nulls        : 8,192  (nulls = employees who stayed — column dropped to avoid leakage)
   After encoding → kept cols  : ['Employee_ID', 'Attrition_Target']
   Target distribut

## 2. Isolate Protected Attributes

**Why this matters mathematically:**  
XGBoost is a greedy ensemble of CART trees. At each node, it evaluates every feature using the gain formula:

$$\text{Gain} = \frac{1}{2}\left[\frac{G_L^2}{H_L+\lambda} + \frac{G_R^2}{H_R+\lambda} - \frac{(G_L+G_R)^2}{H_L+H_R+\lambda}\right] - \gamma$$

where $G$ = sum of first-order loss gradients and $H$ = sum of second-order (Hessian) terms. If `Gender` or `Age` statistically correlates with attrition (even spuriously from historical HR decisions), this gain will be nonzero and the feature **will** be selected. The model encodes the bias into tree weights. Isolating them prevents this entirely while preserving the data for post-hoc fairness audits (e.g., equal FPR/FNR across demographic groups).

In [3]:
# ── Validate protected columns exist ─────────────────────────────────────────
missing_protected = [c for c in PROTECTED_COLS if c not in df.columns]
if missing_protected:
    raise ValueError(f"Protected columns not found: {missing_protected}")

cols_before = list(df.columns)
protected_df = df[PROTECTED_COLS].copy()
model_df     = df.drop(columns=PROTECTED_COLS).copy()

removed = [c for c in cols_before if c not in model_df.columns]
print(f"🛡  Protected attribute isolation")
print(f"   Columns BEFORE : {len(cols_before)}  →  {cols_before}")
print(f"   Removed        : {removed}  (stored in protected_df for fairness audits)")
print(f"   Columns AFTER  : {len(model_df.columns)}  →  {list(model_df.columns)}")

print(f"\n   protected_df preview (first 3 rows):")
print(protected_df.head(3).to_string(index=True))
print(f"\n   Value distribution in protected_df:")
for col in PROTECTED_COLS:
    print(f"   {col}: {protected_df[col].value_counts().to_dict()}")

log_df(model_df, "model_df — after removing protected attributes")

🛡  Protected attribute isolation
   Columns BEFORE : 20  →  ['Age', 'Gender', 'Department', 'Tenure_Years', 'Employment_Type', 'Education_Level', 'Marital_Status', 'Work_Location', 'Job_Satisfaction', 'Work_Life_Balance', 'Management_Support', 'Career_Development', 'Engagement_Level', 'Feedback_Comments', 'Base_Salary', 'Bonus', 'Stock_Options', 'Total_Compensation', 'Compa_Ratio', 'Attrition_Target']
   Removed        : ['Age', 'Gender', 'Marital_Status']  (stored in protected_df for fairness audits)
   Columns AFTER  : 17  →  ['Department', 'Tenure_Years', 'Employment_Type', 'Education_Level', 'Work_Location', 'Job_Satisfaction', 'Work_Life_Balance', 'Management_Support', 'Career_Development', 'Engagement_Level', 'Feedback_Comments', 'Base_Salary', 'Bonus', 'Stock_Options', 'Total_Compensation', 'Compa_Ratio', 'Attrition_Target']

   protected_df preview (first 3 rows):
   Age  Gender Marital_Status
0   38    Male         Single
1   35  Female        Married
2   35  Female        Mar

## 3. NLP Sentiment Extraction via HuggingFace

**Why this matters mathematically:**  
XGBoost splits on scalar thresholds like $x_j < \theta$. A raw string is a non-metric object with no defined ordering — it is impossible to compute a gradient or Hessian over it. A transformer encoder like DistilBERT maps the string into a dense continuous vector via its attention mechanism, then projects the pooled `[CLS]` embedding through a classification head to produce a probability $p \in [0, 1]$. We sign-flip negatives to yield a bipolar scalar $s \in [-1, 1]$, creating a proper real-valued feature the gradient boosting math can act on.

In [5]:
# ── Load DistilBERT pipeline ──────────────────────────────────────────────────
print(f"🤖 Loading sentiment model: {SENTIMENT_MODEL}")
print(f"   This model maps free-text → probability score in (0,1).")
print(f"   We sign-flip negatives so the output is a bipolar scalar in [-1, 1].")
print(f"   POSITIVE confidence → +score  |  NEGATIVE confidence → -score")
sentiment_pipe = pipeline(
    task        = "sentiment-analysis",
    model       = SENTIMENT_MODEL,
    truncation  = True,
    max_length  = 512,
    device      = -1,
    batch_size  = SENTIMENT_BATCH
)

def score_to_signed(result: dict) -> float:
    s = result['score']
    return s if result['label'] == 'POSITIVE' else -s

# ── Pre-inference audit ───────────────────────────────────────────────────────
texts_raw  = model_df[TEXT_COL].fillna('').str.strip()
n_empty    = (texts_raw == '').sum()
n_total    = len(texts_raw)
texts_safe = [t if t else 'no comment provided' for t in texts_raw.tolist()]

print(f"\n📝 Text column audit  [{TEXT_COL}]")
print(f"   Total records        : {n_total:,}")
print(f"   Empty / NaN texts    : {n_empty}  {'✅' if n_empty == 0 else '→ replaced with neutral placeholder'}")
print(f"   Unique comment count : {texts_raw.nunique()}")
print(f"   Sample texts:")
for t in texts_raw.unique()[:3]:
    print(f'     "{t}"')

# ── Batch inference ───────────────────────────────────────────────────────────
print(f"\n⚙  Running inference on {n_total:,} records (batch_size={SENTIMENT_BATCH}) ...")
raw_results = []
for i in tqdm(range(0, len(texts_safe), SENTIMENT_BATCH), desc="Sentiment batches"):
    raw_results.extend(sentiment_pipe(texts_safe[i : i + SENTIMENT_BATCH]))

model_df[SENTIMENT_COL] = [score_to_signed(r) for r in raw_results]

# ── Post-inference audit ──────────────────────────────────────────────────────
scores = model_df[SENTIMENT_COL]
n_pos_sent  = (scores > 0).sum()
n_neg_sent  = (scores < 0).sum()
n_neut_sent = (scores == 0).sum()
print(f"\n✅ Sentiment scores appended as '{SENTIMENT_COL}'")
print(f"   Range        : [{scores.min():.4f}, {scores.max():.4f}]  (expected: [-1, 1])")
print(f"   Mean         : {scores.mean():.4f}  |  Median : {scores.median():.4f}")
print(f"   Positive (>0): {n_pos_sent:,} ({n_pos_sent/n_total:.1%})  — employees with positive sentiment")
print(f"   Negative (<0): {n_neg_sent:,} ({n_neg_sent/n_total:.1%})  — employees with negative sentiment")
print(f"   Neutral (=0) : {n_neut_sent}  (only from empty-text placeholders)")

# ── Drop raw text ──────────────────────────────────────────────────────────────
print(f"\n🗑  Dropping raw text column '{TEXT_COL}'")
print(f"   Reason: XGBoost cannot split on strings. Signal is now captured by {SENTIMENT_COL}.")
model_df = model_df.drop(columns=[TEXT_COL])
print(f"   '{TEXT_COL}' removed  |  '{SENTIMENT_COL}' retained as float")

log_df(model_df, f"model_df — after sentiment extraction, '{TEXT_COL}' replaced by '{SENTIMENT_COL}'")

🤖 Loading sentiment model: distilbert-base-uncased-finetuned-sst-2-english
   This model maps free-text → probability score in (0,1).
   We sign-flip negatives so the output is a bipolar scalar in [-1, 1].
   POSITIVE confidence → +score  |  NEGATIVE confidence → -score


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 1563.15it/s]



📝 Text column audit  [Feedback_Comments]
   Total records        : 10,000
   Empty / NaN texts    : 0  ✅
   Unique comment count : 25
   Sample texts:
     "Salary is decent but bonuses could be better."
     "Work is okay but growth opportunities are limited."
     "Very happy with my role, learning a lot every day."

⚙  Running inference on 10,000 records (batch_size=32) ...


Sentiment batches: 100%|██████████| 313/313 [00:42<00:00,  7.33it/s]



✅ Sentiment scores appended as 'Feedback_Sentiment'
   Range        : [-0.9998, 0.9999]  (expected: [-1, 1])
   Mean         : 0.6660  |  Median : 0.9993
   Positive (>0): 8,385 (83.9%)  — employees with positive sentiment
   Negative (<0): 1,615 (16.2%)  — employees with negative sentiment
   Neutral (=0) : 0  (only from empty-text placeholders)

🗑  Dropping raw text column 'Feedback_Comments'
   Reason: XGBoost cannot split on strings. Signal is now captured by Feedback_Sentiment.
   'Feedback_Comments' removed  |  'Feedback_Sentiment' retained as float

────────────────────────────────────────────────────────────
  📋 model_df — after sentiment extraction, 'Feedback_Comments' replaced by 'Feedback_Sentiment'
────────────────────────────────────────────────────────────
  Shape   : 10,000 rows × 17 columns
  Columns (17):
    • Department                          [str]
    • Tenure_Years                        [float64]
    • Employment_Type                     [str]
    • Education_L

## 4. Categorical Encoding

**Why this matters mathematically:**

**Ordinal encoding** is appropriate when the variable has a *known, meaningful rank order*. For `Education_Level`, assigning integers `{1,2,3,4}` allows the tree to learn thresholds like `Education_Level >= 3` (i.e., Masters or above). The integer gap is irrelevant to CART — only the rank matters for split selection.

**One-Hot Encoding** is appropriate for *nominal* variables with no intrinsic order. If we label-encode `Department` as `{Sales=1, IT=2, HR=3}`, the model falsely assumes `IT(2) > Sales(1)`, which is meaningless and injects false gradient information. OHE converts each category into a binary indicator column, and XGBoost can split on any of them independently.

**Note on `drop_first`:** In logistic regression, dropping one dummy variable prevents perfect multicollinearity. XGBoost (a tree ensemble) is *not* affected by multicollinearity — it's not solving a linear system. Dropping a category removes information and can create split ambiguity. We keep all dummies.

In [6]:
cols_before_encoding = list(model_df.columns)
print(f"🔠 Categorical Encoding")
print(f"   Columns entering this step : {len(cols_before_encoding)}")
print(f"   {cols_before_encoding}\n")

# ── 4.1 Ordinal Encoding ──────────────────────────────────────────────────────
print("─" * 60)
print("  STEP 4.1 — Ordinal Encoding (columns with a natural rank)")
print("─" * 60)
for col, mapping in ORDINAL_MAPS.items():
    if col not in model_df.columns:
        print(f"  ⚠  '{col}' not found — skipping.")
        continue

    original_values = sorted(model_df[col].dropna().unique().tolist())
    unseen = set(original_values) - set(mapping.keys())
    if unseen:
        raise ValueError(
            f"'{col}' contains values not in ORDINAL_MAPS: {unseen}\n"
            f"Add them before continuing."
        )

    model_df[col] = model_df[col].map(mapping)
    print(f"  ✅ '{col}'")
    print(f"     Before : {original_values}")
    print(f"     Mapping: {mapping}")
    print(f"     After  : {sorted(model_df[col].dropna().unique().tolist())}")
    print(f"     Why    : Has a clear hierarchy (High School < Associate < Bachelor < Master < PhD).")
    print(f"              Integer rank lets the tree split on 'Education >= 3' (Bachelor+).")
    print(f"              OHE would create 5 sparse binary columns and lose the rank signal.\n")

# ── 4.2 One-Hot Encoding ──────────────────────────────────────────────────────
print("─" * 60)
print("  STEP 4.2 — One-Hot Encoding (nominal columns, no natural rank)")
print("─" * 60)
existing_nominal = [c for c in NOMINAL_COLS if c in model_df.columns]
missing_nominal  = [c for c in NOMINAL_COLS if c not in model_df.columns]
if missing_nominal:
    print(f"  ⚠  Not found, skipping: {missing_nominal}")

for col in existing_nominal:
    uniq = sorted(model_df[col].dropna().unique().tolist())
    print(f"  📌 '{col}'  ({len(uniq)} categories → {len(uniq)} new binary columns)")
    print(f"     Categories : {uniq}")
    print(f"     Why OHE    : No natural order. Label-encoding would falsely imply rank.")
    print(f"     drop_first : False — XGBoost is a tree model, not logistic regression.")
    print(f"                  Dropping a category removes information; we keep all.\n")

model_df = pd.get_dummies(
    model_df, columns=existing_nominal, drop_first=False, dtype=np.int8
)

cols_after_encoding = list(model_df.columns)
new_cols  = [c for c in cols_after_encoding if c not in cols_before_encoding]
kept_cols = [c for c in cols_before_encoding if c in cols_after_encoding]

print("─" * 60)
print(f"  📊 Encoding summary")
print("─" * 60)
print(f"  Columns BEFORE encoding : {len(cols_before_encoding)}")
print(f"  Columns AFTER  encoding : {len(cols_after_encoding)}")
print(f"  Net new columns added   : +{len(new_cols)}  (OHE dummies)")
print(f"  Columns removed         : -{len(cols_before_encoding) - len(kept_cols)}  "
      f"(original nominal cols replaced by dummies)")
print(f"  New dummy columns       : {new_cols}")

log_df(model_df, "model_df — after all categorical encoding", show_dtypes=True)

🔠 Categorical Encoding
   Columns entering this step : 17
   ['Department', 'Tenure_Years', 'Employment_Type', 'Education_Level', 'Work_Location', 'Job_Satisfaction', 'Work_Life_Balance', 'Management_Support', 'Career_Development', 'Engagement_Level', 'Base_Salary', 'Bonus', 'Stock_Options', 'Total_Compensation', 'Compa_Ratio', 'Attrition_Target', 'Feedback_Sentiment']

────────────────────────────────────────────────────────────
  STEP 4.1 — Ordinal Encoding (columns with a natural rank)
────────────────────────────────────────────────────────────
  ✅ 'Education_Level'
     Before : ['Associate', 'Bachelor', 'High School', 'Master', 'PhD']
     Mapping: {'High School': 1, 'Associate': 2, 'Bachelor': 3, 'Master': 4, 'PhD': 5}
     After  : [1, 2, 3, 4, 5]
     Why    : Has a clear hierarchy (High School < Associate < Bachelor < Master < PhD).
              Integer rank lets the tree split on 'Education >= 3' (Bachelor+).
              OHE would create 5 sparse binary columns and lose t

## 5. Dtype Audit

XGBoost's C++ core requires all input features to be **float32** (or float64). Boolean or object columns cause silent conversion errors or raise exceptions at `.fit()` time. We audit and cast here — before the split — so problems surface early.

In [7]:
feature_cols = [c for c in model_df.columns if c != TARGET_COL]

print(f"🔍 Dtype Audit")
print(f"   Total columns      : {len(model_df.columns)}  (features + target)")
print(f"   Feature columns    : {len(feature_cols)}")
print(f"   Target column      : '{TARGET_COL}' (kept as int, not cast)")
print()

# ── Dtype breakdown before cast ───────────────────────────────────────────────
print("   Dtype breakdown BEFORE cast:")
dtype_groups = model_df[feature_cols].dtypes.astype(str).value_counts()
for dtype, count in dtype_groups.items():
    cols_of_type = model_df[feature_cols].select_dtypes(include=dtype).columns.tolist()
    print(f"   [{dtype:>10}]  {count:>3} column(s): {cols_of_type[:6]}{'...' if len(cols_of_type) > 6 else ''}")

# ── Guard: catch remaining object cols ────────────────────────────────────────
object_cols = model_df[feature_cols].select_dtypes(include='object').columns.tolist()
if object_cols:
    raise ValueError(
        f"\n❌ Object-type columns remain — XGBoost cannot ingest these: {object_cols}\n"
        f"   Fix: add to NOMINAL_COLS (OHE) or ORDINAL_MAPS (ordinal) in Cell 2."
    )
print()
print("   ✅ No object columns remaining — safe to cast.")

# ── Cast to float32 ────────────────────────────────────────────────────────────
model_df[feature_cols] = model_df[feature_cols].astype(np.float32)

print("\n   Dtype breakdown AFTER cast to float32:")
dtype_groups_after = model_df[feature_cols].dtypes.astype(str).value_counts()
for dtype, count in dtype_groups_after.items():
    print(f"   [{dtype:>10}]  {count:>3} column(s)")

print(f"\n   Why float32?")
print(f"   XGBoost's C++ gradient engine expects numeric tensors.")
print(f"   float32 halves memory vs float64 with no loss in tree-split precision.")
print(f"   int8 OHE dummies are safely upcast here.")

log_df(model_df, "model_df — final state before split", show_dtypes=True)

🔍 Dtype Audit
   Total columns      : 34  (features + target)
   Feature columns    : 33
   Target column      : 'Attrition_Target' (kept as int, not cast)

   Dtype breakdown BEFORE cast:
   [      int8]   20 column(s): ['Department_Customer Support', 'Department_Engineering', 'Department_Finance', 'Department_HR', 'Department_Marketing', 'Department_Operations']...
   [   float64]    8 column(s): ['Tenure_Years', 'Job_Satisfaction', 'Work_Life_Balance', 'Management_Support', 'Career_Development', 'Engagement_Level']...
   [     int64]    5 column(s): ['Education_Level', 'Base_Salary', 'Bonus', 'Stock_Options', 'Total_Compensation']

   ✅ No object columns remaining — safe to cast.

   Dtype breakdown AFTER cast to float32:
   [   float32]   33 column(s)

   Why float32?
   XGBoost's C++ gradient engine expects numeric tensors.
   float32 halves memory vs float64 with no loss in tree-split precision.
   int8 OHE dummies are safely upcast here.

────────────────────────────────────────

## 6. Stratified Train/Test Split

**Why stratification is mathematically necessary:**  
With a class imbalance of e.g. 85/15, a naive random 80/20 split has variance in the minority class count. In a small test set, you could end up with 0 positive samples by chance — AUC, F1 and Precision-Recall curves become undefined or misleading. The `stratify=y` argument uses `StratifiedShuffleSplit` internally, which enforces that the ratio $P(Y=1)$ is identical in both train and test — making evaluation metrics reliable.

**Note on splitting the protected dataframe:** `train_test_split` only splits paired arrays if passed together. We use index alignment to correctly slice `protected_df` *after* the main split — no data leakage, no index mismatch.

In [8]:
X = model_df.drop(columns=[TARGET_COL])
y = model_df[TARGET_COL]

print(f"✂  Stratified Train/Test Split  ({1-TEST_SIZE:.0%} / {TEST_SIZE:.0%})")
print(f"   X shape       : {X.shape}")
print(f"   y shape       : {y.shape}")
print(f"   Class balance BEFORE split:")
print(f"     Stayed (0)  : {(y==0).sum():,}  ({(y==0).mean():.2%})")
print(f"     Left   (1)  : {(y==1).sum():,}  ({(y==1).mean():.2%})")
print(f"\n   Why stratify=y?")
print(f"   With 18% attrition, a random split could produce a test set")
print(f"   with a very different ratio by chance. stratify=y enforces the")
print(f"   same 18/82 split in both train and test — making metrics reliable.")

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

# ── Protected split via index alignment ───────────────────────────────────────
# We do NOT pass protected_df into train_test_split — it's unreliable with >2 arrays.
# Instead we use .loc[] on the indices produced by the X/y split.
protected_train = protected_df.loc[X_train.index]
protected_test  = protected_df.loc[X_test.index]

assert (X_train.index == protected_train.index).all(), "❌ Index mismatch: X_train vs protected_train"
assert (X_test.index  == protected_test.index).all(),  "❌ Index mismatch: X_test vs protected_test"

print(f"\n   ✅ Split complete")
print(f"   {'Set':<18} {'Rows':>8}  {'Stayed (0)':>12}  {'Left (1)':>10}  {'Attrition%':>12}")
print(f"   {'─'*18} {'─'*8}  {'─'*12}  {'─'*10}  {'─'*12}")
for name, yy in [("Full dataset", y), ("Train set", y_train), ("Test set", y_test)]:
    print(f"   {name:<18} {len(yy):>8,}  {(yy==0).sum():>12,}  {(yy==1).sum():>10,}  {yy.mean():>11.2%}")

drift = abs(y_train.mean() - y_test.mean())
print(f"\n   Attrition rate drift (train vs test): {drift:.4f}  "
      f"{'✅ negligible' if drift < 0.005 else '⚠ notable — check stratification'}")

print(f"\n   protected_df also split by index alignment (no leakage):")
print(f"   protected_train : {protected_train.shape}  |  protected_test : {protected_test.shape}")
print(f"   Use these ONLY for post-hoc fairness audits — never as model features.")

✂  Stratified Train/Test Split  (80% / 20%)
   X shape       : (10000, 33)
   y shape       : (10000,)
   Class balance BEFORE split:
     Stayed (0)  : 8,192  (81.92%)
     Left   (1)  : 1,808  (18.08%)

   Why stratify=y?
   With 18% attrition, a random split could produce a test set
   with a very different ratio by chance. stratify=y enforces the
   same 18/82 split in both train and test — making metrics reliable.

   ✅ Split complete
   Set                    Rows    Stayed (0)    Left (1)    Attrition%
   ────────────────── ────────  ────────────  ──────────  ────────────
   Full dataset         10,000         8,192       1,808       18.08%
   Train set             8,000         6,554       1,446       18.07%
   Test set              2,000         1,638         362       18.10%

   Attrition rate drift (train vs test): 0.0003  ✅ negligible

   protected_df also split by index alignment (no leakage):
   protected_train : (8000, 3)  |  protected_test : (2000, 3)
   Use these ONLY 

## 7. Calculate `scale_pos_weight`

**Why this is mathematically necessary:**  
XGBoost minimizes the regularized objective:

$$\mathcal{L} = \sum_i l(y_i, \hat{y}_i) + \sum_k \Omega(f_k)$$

where $l$ is log-loss. With 85% negatives, the gradient signal from the minority class is 5.7× weaker. The optimizer converges toward predicting 0 everywhere — achieving high accuracy but zero recall. `scale_pos_weight` scales the loss gradient **and Hessian** for positive instances by a factor $w$:

$$g_i^{\text{scaled}} = w \cdot g_i, \quad h_i^{\text{scaled}} = w \cdot h_i \quad \text{when } y_i = 1$$

The recommended value is $w = N_{\text{neg}} / N_{\text{pos}}$, which restores gradient balance. Computed on **train set only** to prevent test set leakage.

In [9]:
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()

if n_pos == 0:
    raise ValueError("❌ No positive samples in training set — check data or stratification.")

scale_pos_weight = n_neg / n_pos

print(f"⚖  Class Imbalance — scale_pos_weight Calculation")
print(f"\n   Why compute on train set only?")
print(f"   Computing on the full dataset would let information from the test set")
print(f"   influence a training hyperparameter — that is data leakage.")
print()
print(f"   Training set breakdown:")
print(f"     Stayed  (negative, y=0) : {n_neg:,}  ({n_neg/len(y_train):.2%})")
print(f"     Left    (positive, y=1) : {n_pos:,}  ({n_pos/len(y_train):.2%})")
print(f"     Ratio   neg / pos       : {n_neg:,} / {n_pos:,} = {scale_pos_weight:.4f}")
print()
print(f"   What this means:")
print(f"   XGBoost will weight each attrition gradient {scale_pos_weight:.2f}× heavier")
print(f"   than a stay gradient. Without this, the model would predict 'stay'")
print(f"   for everyone and still achieve ~82% accuracy — useless for HR.")
print()
print("═" * 60)
print(f"  ✅  scale_pos_weight  =  {scale_pos_weight:.4f}")
print("═" * 60)
print()
print(f"  Use in XGBoost:")
print(f"    XGBClassifier(scale_pos_weight={scale_pos_weight:.4f}, ...)")
print()
print(f"  Use in Optuna (fix this — do NOT tune it):")
print(f"    params['scale_pos_weight'] = {scale_pos_weight:.4f}")

⚖  Class Imbalance — scale_pos_weight Calculation

   Why compute on train set only?
   Computing on the full dataset would let information from the test set
   influence a training hyperparameter — that is data leakage.

   Training set breakdown:
     Stayed  (negative, y=0) : 6,554  (81.92%)
     Left    (positive, y=1) : 1,446  (18.07%)
     Ratio   neg / pos       : 6,554 / 1,446 = 4.5325

   What this means:
   XGBoost will weight each attrition gradient 4.53× heavier
   than a stay gradient. Without this, the model would predict 'stay'
   for everyone and still achieve ~82% accuracy — useless for HR.

════════════════════════════════════════════════════════════
  ✅  scale_pos_weight  =  4.5325
════════════════════════════════════════════════════════════

  Use in XGBoost:
    XGBClassifier(scale_pos_weight=4.5325, ...)

  Use in Optuna (fix this — do NOT tune it):
    params['scale_pos_weight'] = 4.5325


## 8. Final Outputs Summary

All arrays are ready to pass into XGBoost or Optuna.

In [10]:
print("━" * 60)
print("  ✅  PIPELINE COMPLETE — ALL OBJECTS READY FOR XGBOOST / OPTUNA")
print("━" * 60)

print(f"""
  OBJECT              SHAPE             DESCRIPTION
  ──────────────────────────────────────────────────────────
  X_train         {str(X_train.shape):<18}float32 features, train set
  X_test          {str(X_test.shape):<18}float32 features, test set
  y_train         {str(y_train.shape):<18}int target, stratified
  y_test          {str(y_test.shape):<18}int target, stratified
  protected_train {str(protected_train.shape):<18}Age/Gender/Marital — fairness audit only
  protected_test  {str(protected_test.shape):<18}Age/Gender/Marital — fairness audit only
  scale_pos_weight  {scale_pos_weight:.4f}            pass directly to XGBClassifier
""")

print(f"  Feature columns ({len(X_train.columns)}):")
for col in X_train.columns:
    print(f"    • {col}")

print(f"""
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  FULL PIPELINE COLUMN JOURNEY SUMMARY
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  Started with (post-merge, 20 cols):
    Age, Gender, Department, Tenure_Years, Employment_Type,
    Education_Level, Marital_Status, Work_Location,
    Job_Satisfaction, Work_Life_Balance, Management_Support,
    Career_Development, Engagement_Level, Feedback_Comments,
    Base_Salary, Bonus, Stock_Options, Total_Compensation,
    Compa_Ratio, Attrition_Target

  Dropped — not features:
    ✗ Employee_ID   (identifier, no predictive value)
    ✗ Role          (33 categories, signal captured by Compa_Ratio + Department)

  Isolated — fairness audit only:
    ✗ Age, Gender, Marital_Status  → stored in protected_df

  Transformed:
    ✗ Feedback_Comments (text)     → ✓ Feedback_Sentiment (float, DistilBERT)
    ✗ Education_Level (string)     → ✓ Education_Level (int 1–5, ordinal)
    ✗ Department (8 categories)    → ✓ 8 binary dummy columns
    ✗ Employment_Type (3 cats)     → ✓ 3 binary dummy columns
    ✗ Work_Location (9 cities)     → ✓ 9 binary dummy columns

  Final X shape: {X_train.shape[1]} features × {len(X_train)+len(X_test):,} rows
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
""")

print("  Optuna quickstart:")
print("""
import xgboost as xgb, optuna
from sklearn.metrics import average_precision_score

def objective(trial):
    params = {
        'max_depth':        trial.suggest_int('max_depth', 3, 9),
        'learning_rate':    trial.suggest_float('learning_rate', 1e-3, 0.3, log=True),
        'n_estimators':     trial.suggest_int('n_estimators', 100, 1000),
        'subsample':        trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 1, 10),
        'gamma':            trial.suggest_float('gamma', 0, 5),
        'reg_alpha':        trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':       trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'scale_pos_weight': scale_pos_weight,   # fixed — do NOT tune
        'eval_metric':      'aucpr',            # PR-AUC > ROC-AUC for imbalanced data
        'use_label_encoder': False,
        'random_state':     42,
    }
    model = xgb.XGBClassifier(**params)
    model.fit(X_train, y_train,
              eval_set=[(X_test, y_test)],
              early_stopping_rounds=50, verbose=False)
    return average_precision_score(y_test, model.predict_proba(X_test)[:, 1])

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=200)
""")

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ✅  PIPELINE COMPLETE — ALL OBJECTS READY FOR XGBOOST / OPTUNA
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

  OBJECT              SHAPE             DESCRIPTION
  ──────────────────────────────────────────────────────────
  X_train         (8000, 33)        float32 features, train set
  X_test          (2000, 33)        float32 features, test set
  y_train         (8000,)           int target, stratified
  y_test          (2000,)           int target, stratified
  protected_train (8000, 3)         Age/Gender/Marital — fairness audit only
  protected_test  (2000, 3)         Age/Gender/Marital — fairness audit only
  scale_pos_weight  4.5325            pass directly to XGBClassifier

  Feature columns (33):
    • Tenure_Years
    • Education_Level
    • Job_Satisfaction
    • Work_Life_Balance
    • Management_Support
    • Career_Development
    • Engagement_Level
    • Base_Salary
    • Bonus
    • Stock_Opti